In [2]:
import pandas as pd
import snowflake.connector
import json


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("SNOWFLAKE_USER"))
print(os.getenv("SNOWFLAKE_ACCOUNT"))

ALEXANDRELOUMI
UZMURSA-ML30769


In [4]:
conn = snowflake.connector.connect(
    user = os.getenv("SNOWFLAKE_USER"),
    password = os.getenv("SNOWFLAKE_PASSWORD"),
    account = os.getenv("SNOWFLAKE_ACCOUNT")
)

In [5]:
cursor = conn.cursor()
cursor.execute("SELECT CURRENT_VERSION()")
cursor.fetchone()

('10.34.101',)

In [6]:
cursor.execute("SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE()")
cursor.fetchone()

(None, None, 'COMPUTE_WH')

In [7]:
cursor.execute("SHOW DATABASES")
cursor.fetchall()

[(datetime.datetime(2026, 9, 8, 9, 38, 10, 306000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>),
  'ANALYTICS',
  'N',
  'N',
  '',
  'ACCOUNTADMIN',
  '',
  '',
  '1',
  'STANDARD',
  'ROLE',
  None,
  None,
  None),
 (datetime.datetime(2026, 9, 11, 6, 1, 53, 118000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>),
  'NHL_ANALYTICS',
  'N',
  'N',
  '',
  'ACCOUNTADMIN',
  '',
  '',
  '1',
  'STANDARD',
  'ROLE',
  None,
  None,
  None),
 (datetime.datetime(2026, 9, 8, 9, 38, 9, 466000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>),
  'RAW',
  'N',
  'N',
  '',
  'ACCOUNTADMIN',
  '',
  '',
  '1',
  'STANDARD',
  'ROLE',
  None,
  None,
  None),
 (datetime.datetime(2026, 9, 8, 9, 33, 10, 765000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>),
  'SNOWFLAKE',
  'N',
  'N',
  'SNOWFLAKE.ACCOUNT_USAGE',
  '',
  '',
  '',
  '0',
  'APPLICATION',
  '',
  None,
  None,
  None),
 (datetime.datetime(2026, 9, 8, 9, 33

In [8]:
cursor.execute("USE DATABASE NHL_ANALYTICS")
cursor.execute("SELECT CURRENT_DATABASE()")
cursor.fetchone()

('NHL_ANALYTICS',)

In [9]:
cursor.execute("CREATE SCHEMA IF NOT EXISTS RAW")
cursor.fetchall()

[('RAW already exists, statement succeeded.',)]

In [10]:
cursor.execute("SHOW SCHEMAS LIKE 'RAW'")
cursor.fetchall()

[(datetime.datetime(2026, 9, 11, 6, 2, 9, 944000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>),
  'RAW',
  'N',
  'N',
  'NHL_ANALYTICS',
  'ACCOUNTADMIN',
  '',
  '',
  '1',
  'ROLE',
  None,
  None,
  None,
  None,
  'false')]

In [11]:
cursor.execute("USE SCHEMA RAW")

In [15]:
cursor.execute("CREATE STAGE IF NOT EXISTS NHL_GAMES_STAGE")

In [18]:
cursor.execute("PUT 'file://C:/Users/alexa/Desktop/NHL PROJECT/nhl_analytics/data/raw/games.json' @NHL_GAMES_STAGE")

In [19]:
cursor.execute("LIST @NHL_GAMES_STAGE")
cursor.fetchall()

[('nhl_games_stage/games.json.gz',
  290272,
  'db1097a0d25e543ef29a6ce0b8b0148c',
  'Mon, 14 Sep 2026 13:10:03 GMT')]

In [21]:
cursor.execute("CREATE TABLE IF NOT EXISTS NHL_ANALYTICS.RAW.RAW_GAMES (game_id NUMBER, data VARIANT)")

In [ ]:
with open("../data/raw/games.json", "r", encoding="utf-8") as f:
    games = json.load(f)

In [23]:
cursor.execute("CREATE STAGE IF NOT EXISTS NHL_PLAYERS_STAGE")

In [27]:
cursor.execute("PUT 'file://C:/Users/alexa/Desktop/NHL PROJECT/nhl_analytics/data/raw/players.json' @NHL_PLAYERS_STAGE")

In [29]:
cursor.execute("LIST @NHL_PLAYERS_STAGE")
cursor.fetchall()

[('nhl_players_stage/players.json.gz',
  24192,
  '018533a7ef21d4eb60d59a26d16eddbe',
  'Fri, 25 Sep 2026 13:51:09 GMT')]

In [30]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS NHL_ANALYTICS.RAW.RAW_PLAYERS (
player_id NUMBER,
data VARIANT
)
""")

In [32]:
cursor.execute("""
COPY INTO NHL_ANALYTICS.RAW.RAW_PLAYERS (player_id, data)
FROM (
    SELECT
        $1:id::NUMBER,
        $1
    FROM @NHL_PLAYERS_STAGE
)
FILE_FORMAT = (
    TYPE = 'JSON'
)
""")

In [36]:
cursor.execute("TRUNCATE TABLE NHL_ANALYTICS.RAW.RAW_PLAYERS")

In [40]:
with open("../data/raw/players.json", "r", encoding="utf-8") as f:
    players = json.load(f)

In [41]:
len(players)

701

In [42]:
for player in players:
    cursor.execute(
        """
        INSERT INTO NHL_ANALYTICS.RAW.RAW_PLAYERS (player_id, data)
        SELECT %s, PARSE_JSON(%s)
        """,
        (player["id"], json.dumps(player))
    )

In [43]:
cursor.execute("SELECT COUNT(*) FROM NHL_ANALYTICS.RAW.RAW_PLAYERS")
cursor.fetchone()

(701,)

In [44]:
cursor.execute("""
SELECT player_id, data
FROM NHL_ANALYTICS.RAW.RAW_PLAYERS
LIMIT 5
""")

cursor.fetchall()

[(8480313,
  '{\n  "birth_city": "Calgary",\n  "birth_country": "CAN",\n  "birth_date": "1997-02-25",\n  "first_name": "Logan",\n  "height": 193,\n  "id": 8480313,\n  "last_name": "Thompson",\n  "number": 48,\n  "position": "G",\n  "shoots_catches": "R",\n  "team": "WSH",\n  "weight": 94\n}'),
 (8483532,
  '{\n  "birth_city": "Drayton Valley",\n  "birth_country": "CAN",\n  "birth_date": "1999-03-03",\n  "first_name": "Clay",\n  "height": 193,\n  "id": 8483532,\n  "last_name": "Stevenson",\n  "number": 33,\n  "position": "G",\n  "shoots_catches": "L",\n  "team": "WSH",\n  "weight": 88\n}'),
 (8479292,
  '{\n  "birth_city": "Lakeville",\n  "birth_country": "USA",\n  "birth_date": "1993-12-18",\n  "first_name": "Charlie",\n  "height": 188,\n  "id": 8479292,\n  "last_name": "Lindgren",\n  "number": 79,\n  "position": "G",\n  "shoots_catches": "R",\n  "team": "WSH",\n  "weight": 86\n}'),
 (8480873,
  '{\n  "birth_city": "Uppsala",\n  "birth_country": "SWE",\n  "birth_date": "2000-03-07",\n 

In [34]:
cursor.execute("""
SELECT player_id, data
FROM NHL_ANALYTICS.RAW.RAW_PLAYERS
LIMIT 5
""")
cursor.fetchall()

[(None,
  '[\n  {\n    "birth_city": "Calgary",\n    "birth_country": "CAN",\n    "birth_date": "1997-02-25",\n    "first_name": "Logan",\n    "height": 193,\n    "id": 8480313,\n    "last_name": "Thompson",\n    "number": 48,\n    "position": "G",\n    "shoots_catches": "R",\n    "team": "WSH",\n    "weight": 94\n  },\n  {\n    "birth_city": "Drayton Valley",\n    "birth_country": "CAN",\n    "birth_date": "1999-03-03",\n    "first_name": "Clay",\n    "height": 193,\n    "id": 8483532,\n    "last_name": "Stevenson",\n    "number": 33,\n    "position": "G",\n    "shoots_catches": "L",\n    "team": "WSH",\n    "weight": 88\n  },\n  {\n    "birth_city": "Lakeville",\n    "birth_country": "USA",\n    "birth_date": "1993-12-18",\n    "first_name": "Charlie",\n    "height": 188,\n    "id": 8479292,\n    "last_name": "Lindgren",\n    "number": 79,\n    "position": "G",\n    "shoots_catches": "R",\n    "team": "WSH",\n    "weight": 86\n  },\n  {\n    "birth_city": "Uppsala",\n    "birth_count

In [45]:
cursor.execute("""
SELECT *
FROM NHL_ANALYTICS.RAW_STAGING.STG_PLAYERS
LIMIT 5
""")

cursor.fetchall()

[(8480313,
  'Logan',
  'Thompson',
  48,
  'G',
  'R',
  193,
  94,
  datetime.date(1997, 2, 25),
  'Calgary',
  'CAN',
  'WSH'),
 (8483532,
  'Clay',
  'Stevenson',
  33,
  'G',
  'L',
  193,
  88,
  datetime.date(1999, 3, 3),
  'Drayton Valley',
  'CAN',
  'WSH'),
 (8479292,
  'Charlie',
  'Lindgren',
  79,
  'G',
  'R',
  188,
  86,
  datetime.date(1993, 12, 18),
  'Lakeville',
  'USA',
  'WSH'),
 (8480873,
  'Rasmus',
  'Sandin',
  38,
  'D',
  'L',
  180,
  86,
  datetime.date(2000, 3, 7),
  'Uppsala',
  'SWE',
  'WSH'),
 (8478911,
  'Matt',
  'Roy',
  3,
  'D',
  'R',
  188,
  100,
  datetime.date(1995, 3, 1),
  'Detroit',
  'USA',
  'WSH')]

In [ ]:
len(games)

1394

In [ ]:
games[0]

{'id': 2025020001,
 'season': 20252026,
 'gameType': 2,
 'venue': {'default': 'Amerant Bank Arena'},
 'neutralSite': False,
 'startTimeUTC': '2025-10-07T21:00:00Z',
 'easternUTCOffset': '-04:00',
 'venueUTCOffset': '-04:00',
 'venueTimezone': 'US/Eastern',
 'gameState': 'OFF',
 'gameScheduleState': 'OK',
 'tvBroadcasts': [{'id': 309,
   'market': 'N',
   'countryCode': 'US',
   'network': 'ESPN',
   'sequenceNumber': 10},
  {'id': 284,
   'market': 'N',
   'countryCode': 'CA',
   'network': 'SN1',
   'sequenceNumber': 113},
  {'id': 281,
   'market': 'N',
   'countryCode': 'CA',
   'network': 'TVAS',
   'sequenceNumber': 120}],
 'awayTeam': {'id': 16,
  'commonName': {'default': 'Blackhawks'},
  'placeName': {'default': 'Chicago'},
  'placeNameWithPreposition': {'default': 'Chicago', 'fr': 'de Chicago'},
  'abbrev': 'CHI',
  'logo': 'https://assets.nhle.com/logos/nhl/svg/CHI_light.svg?season=20252026',
  'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/CHI_dark.svg?season=20252026',


In [ ]:
games[0].keys()

dict_keys(['id', 'season', 'gameType', 'venue', 'neutralSite', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'venueTimezone', 'gameState', 'gameScheduleState', 'tvBroadcasts', 'awayTeam', 'homeTeam', 'periodDescriptor', 'gameOutcome', 'winningGoalie', 'winningGoalScorer', 'threeMinRecap', 'threeMinRecapFr', 'gameCenterLink'])

In [ ]:
games[0]["homeTeam"]

{'id': 13,
 'commonName': {'default': 'Panthers'},
 'placeName': {'default': 'Florida', 'fr': 'Floride'},
 'placeNameWithPreposition': {'default': 'Florida', 'fr': 'de la Floride'},
 'abbrev': 'FLA',
 'logo': 'https://assets.nhle.com/logos/nhl/svg/FLA_light.svg',
 'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/FLA_dark.svg',
 'homeSplitSquad': False,
 'score': 3}

In [ ]:
games[0]["awayTeam"]

{'id': 16,
 'commonName': {'default': 'Blackhawks'},
 'placeName': {'default': 'Chicago'},
 'placeNameWithPreposition': {'default': 'Chicago', 'fr': 'de Chicago'},
 'abbrev': 'CHI',
 'logo': 'https://assets.nhle.com/logos/nhl/svg/CHI_light.svg?season=20252026',
 'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/CHI_dark.svg?season=20252026',
 'awaySplitSquad': False,
 'score': 2}

In [ ]:
games[0]["winningGoalie"]

{'playerId': 8475683,
 'firstInitial': {'default': 'S.'},
 'lastName': {'default': 'Bobrovsky',
  'cs': 'Bobrovskij',
  'fi': 'Bobrovski',
  'sk': 'Bobrovskij'}}

In [ ]:
games[0]["gameOutcome"]

{'lastPeriodType': 'REG'}

In [ ]:
game = games[0]
print(game["id"])
print(game["season"])
print(game["startTimeUTC"])
print(game["homeTeam"]["id"])
print(game["awayTeam"]["id"])
print(game["homeTeam"]["score"])
print(game["awayTeam"]["score"])


2025020001
20252026
2025-10-07T21:00:00Z
13
16
3
2


In [ ]:
games_rows = []


for game in games:

    game_row = {
        "game_id" : game["id"],
        "season" : game["season"],
        "game_type" : game["gameType"],
        "game_datetime_utc" : game["startTimeUTC"],
        "home_team_id" : game["homeTeam"]["id"],
        "away_team_id" : game["awayTeam"]["id"],
        "home_score" : game["homeTeam"]["score"],
        "away_score" : game["awayTeam"]["score"],
    }

    games_rows.append(game_row)

In [ ]:
games_df = pd.DataFrame(games_rows)

In [ ]:
games_df.head()

,game_id,season,game_type,game_datetime_utc,home_team_id,away_team_id,home_score,away_score
0,2025020001,20252026,2,2025-10-07T21:00:00Z,13,16,3,2
1,2025020002,20252026,2,2025-10-08T00:00:00Z,3,5,0,3
2,2025020003,20252026,2,2025-10-08T02:30:00Z,26,21,1,4
3,2025020004,20252026,2,2025-10-08T23:00:00Z,10,8,5,2
4,2025020005,20252026,2,2025-10-08T23:30:00Z,15,6,1,3


In [ ]:
games_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1394 entries, 0 to 1393
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   game_id            1394 non-null   int64
 1   season             1394 non-null   int64
 2   game_type          1394 non-null   int64
 3   game_datetime_utc  1394 non-null   str  
 4   home_team_id       1394 non-null   int64
 5   away_team_id       1394 non-null   int64
 6   home_score         1394 non-null   int64
 7   away_score         1394 non-null   int64
dtypes: int64(7), str(1)
memory usage: 87.3 KB


In [ ]:
games_df["game_type"].value_counts()

game_type
2    1312
3      82
Name: count, dtype: int64

In [ ]:
games_df["game_datetime_utc"] = pd.to_datetime(games_df["game_datetime_utc"])

In [ ]:
games_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1394 entries, 0 to 1393
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   game_id            1394 non-null   int64              
 1   season             1394 non-null   int64              
 2   game_type          1394 non-null   int64              
 3   game_datetime_utc  1394 non-null   datetime64[us, UTC]
 4   home_team_id       1394 non-null   int64              
 5   away_team_id       1394 non-null   int64              
 6   home_score         1394 non-null   int64              
 7   away_score         1394 non-null   int64              
dtypes: datetime64[us, UTC](1), int64(7)
memory usage: 87.3 KB


In [ ]:
games_df["game_id"].nunique()

1394

In [ ]:
games_df[["home_score", "away_score"]].isna().sum()

home_score    0
away_score    0
dtype: int64

In [ ]:
games_df[["home_score", "away_score"]].min()

home_score    0
away_score    0
dtype: int64

In [ ]:
pd.concat([
    games_df["home_team_id"],
    games_df["away_team_id"]
]).nunique()

32

In [ ]:
team_rows = []

for game in games:
    team_rows.append(game["homeTeam"])
    team_rows.append(game["awayTeam"])

print(len(team_rows))

2788


In [ ]:
print(team_rows[0])

{'id': 13, 'commonName': {'default': 'Panthers'}, 'placeName': {'default': 'Florida', 'fr': 'Floride'}, 'placeNameWithPreposition': {'default': 'Florida', 'fr': 'de la Floride'}, 'abbrev': 'FLA', 'logo': 'https://assets.nhle.com/logos/nhl/svg/FLA_light.svg', 'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/FLA_dark.svg', 'homeSplitSquad': False, 'score': 3}


In [ ]:
team_rows_clean = []

for team in team_rows:
    team_row = {
        "team_id": team["id"],
        "team_name": team["commonName"]["default"],
        "city": team["placeName"]["default"],
        "abbreviation": team["abbrev"]
    }

    team_rows_clean.append(team_row)

In [ ]:
team_df = pd.DataFrame(team_rows_clean)

team_df.head()

,team_id,team_name,city,abbreviation
0,13,Panthers,Florida,FLA
1,16,Blackhawks,Chicago,CHI
2,3,Rangers,New York,NYR
3,5,Penguins,Pittsburgh,PIT
4,26,Kings,Los Angeles,LAK


In [ ]:
team_df = team_df.drop_duplicates(subset = "team_id")

team_df.shape

(32, 4)

In [ ]:
game_team_ids = pd.concat([
    games_df["home_team_id"],
    games_df["away_team_id"]
])

In [ ]:
game_team_ids.nunique()

32

In [ ]:
team_df.isna().sum()

team_id         0
team_name       0
city            0
abbreviation    0
dtype: int64

In [ ]:
team_df.dtypes

team_id         int64
team_name         str
city              str
abbreviation      str
dtype: object

In [ ]:
team_df["team_id"].duplicated().sum()

np.int64(0)

In [ ]:
games_df["game_id"].duplicated().sum()
games_df[["home_score", "away_score"]].isna().sum()
games_df["game_datetime_utc"].isna().sum()
games_df.dtypes

game_id                            int64
season                             int64
game_type                          int64
game_datetime_utc    datetime64[us, UTC]
home_team_id                       int64
away_team_id                       int64
home_score                         int64
away_score                         int64
dtype: object